# GéoMarketing IDF : J11 — Cartographie et contrôles géographiques en Île-de-France

## Objectifs
- Associer le profil communal aux contours géographiques.
- Contrôler les coordonnées des observations détaillées.
- Comparer leur position à la commune déclarée.
- Produire des diagnostics et des cartes interactives.

## Principes
- Conserver les codes communaux déclarés.
- Signaler les écarts sans réaffectation automatique.
- Conserver les observations sans coordonnées dans les comptages.
- Documenter la source et la date de récupération des contours.

## Limites
- Les contours utilisés constituent une référence cartographique.
- Une position dans le bon contour ne valide pas l'adresse exacte.
- Une distance géométrique n'est pas une distance de marche.
- Les cartes ne constituent pas encore un score d'implantation.

In [ ]:
%pip install plotly geopandas nbformat

In [ ]:
#Import librairies

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

from pathlib import Path
from datetime import datetime, timezone
from html import escape
import hashlib
import json
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly
import plotly.graph_objects as go
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
CONFIG_PLOTLY = {'scrollZoom': True, 'displaylogo': False}
print('Plotly :', plotly.__version__)
print('GeoPandas :', gpd.__version__)


In [ ]:
RACINE = Path(os.environ.get('GEOMARKETING_RACINE',
              r'C:\Users\almou\OneDrive\GeoMarketing_IDF'))
FICHIER_COMMUNES = RACINE / 'data/processed/j11/profil_communes_idf_j11.gpkg'
FICHIER_RESTAURANTS = RACINE / 'data/processed/j11/restaurants_points.gpkg'
FICHIER_DETAIL_RESTAURANTS = RACINE / 'data/processed/restaurants_idf_detail_j9b.csv'  # Facultatif
DOSSIER_BPE_NOMS = RACINE / 'data/raw/bpe'
FICHIER_BPE_NOMS = DOSSIER_BPE_NOMS / 'restaurants_idf_2025.csv'  # Adapter si nécessaire (CSV ou ZIP)
COLONNE_LIBELLE_RESTAURANT = None  # Nom exact d'un champ personnalisé, si nécessaire
AFFICHER_NOMS_RESTAURANTS = False  # False : noms seulement au survol
DOSSIER_CARTES = RACINE / 'outputs/cartes/j11'
DOSSIER_INTERIM = RACINE / 'data/interim/j11/plotly'
for dossier in [DOSSIER_CARTES, DOSSIER_INTERIM]:
    dossier.mkdir(parents=True, exist_ok=True)
print('Profil géographique :', FICHIER_COMMUNES)


In [ ]:
def exiger_colonnes(table, colonnes, nom):
    absentes = set(colonnes) - set(table.columns)
    if absentes:
        raise ValueError(f'{nom} : colonnes absentes {sorted(absentes)}. '
                         f'Disponibles : {table.columns.tolist()}')


def texte(valeur, secours='Non renseigné'):
    if valeur is None or pd.isna(valeur) or not str(valeur).strip():
        return secours
    return str(valeur).strip()


def nombres(serie):
    valeurs = pd.to_numeric(
        serie.astype('string').str.strip()
        .str.replace('\u202f', '', regex=False)
        .str.replace('\u00a0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce',
    )
    tableau = valeurs.to_numpy(dtype=float, na_value=np.nan, copy=True)
    tableau[~np.isfinite(tableau)] = np.nan
    return pd.Series(tableau, index=serie.index)


def empreinte(chemin):
    resultat = hashlib.sha256()
    with chemin.open('rb') as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b''):
            resultat.update(bloc)
    return resultat.hexdigest()


def enregistrer_csv(table, chemin):
    table.to_csv(chemin, index=False, encoding='utf-8-sig')


def lire_couche(chemin, nom, types, layer=None):
    if not chemin.is_file():
        raise FileNotFoundError(f'{nom} : fichier absent {chemin}')
    options = {'layer': layer} if layer is not None else {}
    couche = gpd.read_file(chemin, **options)
    if couche.empty or couche.crs is None:
        raise ValueError(f'{nom} : couche vide ou CRS absent.')
    valide = (couche.geometry.notna() & ~couche.geometry.is_empty
              & couche.geometry.is_valid & couche.geometry.geom_type.isin(types))
    if not valide.all():
        raise ValueError(f'{nom} : {int((~valide).sum())} géométries à vérifier '
                         'dans le notebook de préparation avant de cartographier.')
    return couche.to_crs(2154)


def preparer_communes(couche):
    exiger_colonnes(couche, ['CODGEO'], 'Communes')
    couche = couche.copy()
    codes = (couche['CODGEO'].astype('string').str.strip()
             .str.replace(r'\.0$', '', regex=True))
    codes = codes.where(codes.str.fullmatch(r'\d{1,5}', na=False)).str.zfill(5)
    if codes.isna().any() or not codes.is_unique:
        raise ValueError('Codes communaux absents, invalides ou répétés.')
    couche['CODGEO'] = codes
    noms = pd.Series(pd.NA, index=couche.index, dtype='string')
    for colonne in ['NOM_COMMUNE_CONTOUR', 'NOM_COMMUNE', 'LIBELLE']:
        if colonne in couche.columns:
            candidat = couche[colonne].astype('string').str.strip().replace('', pd.NA)
            noms = noms.fillna(candidat)
    couche['NOM_CARTE'] = noms.fillna(codes)
    return couche


def contours_geojson(communes, tolerance_metres=20):
    affichage = communes[['CODGEO', 'geometry']].to_crs(2154).copy()
    if tolerance_metres:
        affichage.geometry = affichage.geometry.simplify(
            tolerance_metres, preserve_topology=True)
    affichage = affichage.to_crs(4326)
    return json.loads(affichage.to_json(na='null', drop_id=True))


def ajouter_contours(fig, communes, couleur='#f1f5f9', nom='Limites communales',
                     largeur=0.5, tolerance_metres=20):
    fig.add_trace(go.Choropleth(
        geojson=contours_geojson(communes, tolerance_metres),
        locations=communes['CODGEO'].tolist(),
        featureidkey='properties.CODGEO',
        z=[0] * len(communes), zmin=0, zmax=1,
        colorscale=[[0, couleur], [1, couleur]],
        showscale=False, showlegend=False, name=nom,
        marker_line_color='#64748b', marker_line_width=largeur,
        text=[escape(texte(nom_commune)) for nom_commune in communes['NOM_CARTE']],
        hovertemplate='%{text}<br>Code commune : %{location}<extra></extra>',
    ))
    return fig


def configurer_figure(fig, titre, emprise, source):
    xmin, ymin, xmax, ymax = emprise.to_crs(4326).total_bounds
    marge_x = max((xmax - xmin) * 0.04, 0.001)
    marge_y = max((ymax - ymin) * 0.04, 0.001)
    # Emprise fixe : masquer une ligne dans la légende ne recentre pas la carte.
    fig.update_geos(
        visible=False, bgcolor='white', projection_type='mercator',
        fitbounds=False, center={'lon': (xmin + xmax) / 2, 'lat': (ymin + ymax) / 2},
        lonaxis={'range': [xmin - marge_x, xmax + marge_x], 'showgrid': False},
        lataxis={'range': [ymin - marge_y, ymax + marge_y], 'showgrid': False},
    )
    fig.update_layout(
        title={'text': titre, 'x': 0.02}, height=800,
        paper_bgcolor='white', template='plotly_white',
        margin={'l': 10, 'r': 10, 't': 75, 'b': 80},
        legend={'title': {'text': 'Couches — cliquer pour masquer'},
                'groupclick': 'toggleitem'},
        annotations=[{'text': source, 'xref': 'paper', 'yref': 'paper',
                      'x': 0, 'y': -0.07, 'xanchor': 'left', 'showarrow': False,
                      'font': {'size': 11, 'color': '#475569'}}],
    )
    return fig


def afficher_et_exporter(fig, chemin):
    # Inclut Plotly.js dans le HTML : aucun compte Plotly ni token.
    fig.write_html(str(chemin), include_plotlyjs=True, full_html=True,
                   config=CONFIG_PLOTLY)
    fig.show(config=CONFIG_PLOTLY)
    print('Carte enregistrée :', chemin)


def selectionner_secteur(communes, codes, marge_metres=3000):
    absents = set(codes) - set(communes['CODGEO'])
    if absents:
        raise ValueError(f'Communes absentes : {sorted(absents)}')
    selection = communes[communes['CODGEO'].isin(codes)].copy()
    contour = selection.dissolve().geometry.iloc[0]
    zone = gpd.GeoDataFrame({'ZONE': ['Affichage']},
                           geometry=[contour.buffer(marge_metres)], crs=2154)
    contexte = gpd.clip(communes, zone, keep_geom_type=True)
    return selection, zone, contexte


from contextlib import ExitStack
from pathlib import Path
from zipfile import ZipFile
import csv

import pandas as pd


COLONNES_NOMS_RESTAURANTS = [
    'NOM_RESTAURANT_CARTE', 'LIBELLE_RESTAURANT', 'LIBELLE_RESTAURANT_CARTE',
    'NOM_RESTAURANT', 'NOM_AFFICHAGE', 'ENSEIGNE', 'ENSEIGNE1ETABLISSEMENT',
    'NOM_COMMERCIAL', 'DENOMINATION_USUELLE',
    'DENOMINATIONUSUELLEETABLISSEMENT', 'NOMRS', 'RAISON_SOCIALE', 'CNOMRS',
]


def nettoyer_libelle_restaurant(serie):
    absences = {'', 'nan', 'none', '<na>', 'null', 'nd', '[nd]', 'n/d', 'n/a', '-',
                '_z', '_u', '_n', 'inconnu', 'non renseigné', 'non renseigne',
                'non diffusé', 'non diffuse', 'nom non disponible',
                'libellé absent dans la source'}
    valeurs = serie.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    return valeurs.mask(valeurs.str.casefold().isin(absences))


def choisir_noms_restaurants(table, colonne_libelle=None):
    table = table.rename(columns=lambda c: str(c).strip().upper())
    if table.columns.duplicated().any():
        raise ValueError('Noms de colonnes répétés après normalisation.')
    candidats = list(COLONNES_NOMS_RESTAURANTS)
    if colonne_libelle:
        candidats.insert(0, str(colonne_libelle).strip().upper())
    noms = pd.Series(pd.NA, index=table.index, dtype='string')
    colonnes = pd.Series(pd.NA, index=table.index, dtype='string')
    for colonne in dict.fromkeys(candidats):
        if colonne not in table.columns:
            continue
        candidat = nettoyer_libelle_restaurant(table[colonne])
        provenance = pd.Series(colonne, index=table.index, dtype='string')
        # CNOMRS est un complément : conserver les deux informations lorsqu'elles diffèrent.
        if colonne == 'NOMRS' and 'CNOMRS' in table.columns:
            complement = nettoyer_libelle_restaurant(table['CNOMRS'])
            ajouter = (candidat.notna() & complement.notna()
                       & candidat.str.casefold().ne(complement.str.casefold()).fillna(False))
            candidat.loc[ajouter] = candidat.loc[ajouter] + ' — ' + complement.loc[ajouter]
            provenance.loc[ajouter] = 'NOMRS+CNOMRS'
        utiliser = noms.isna() & candidat.notna()
        noms.loc[utiliser] = candidat.loc[utiliser]
        colonnes.loc[utiliser] = provenance.loc[utiliser]
    return noms, colonnes


def normaliser_siret_pour_noms(serie):
    valeurs = serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    # Ne pas transformer une valeur invalide en identifiant par suppression ou ajout de chiffres.
    return valeurs.where(valeurs.str.fullmatch(r'\d{14}', na=False)
                         & valeurs.ne('00000000000000'))


def siret_des_restaurants(table):
    table = table.rename(columns=lambda c: str(c).strip().upper())
    resultat = pd.Series(pd.NA, index=table.index, dtype='string')
    if 'SIRET' in table.columns:
        resultat = normaliser_siret_pour_noms(table['SIRET'])
    for colonne in ['CLE_EQUIPEMENT', 'ID_OBSERVATION']:
        if colonne not in table.columns:
            continue
        # Seuls les formats explicitement identifiables sont interprétés comme un SIRET.
        valeurs = table[colonne].astype('string').str.strip()
        extrait = valeurs.str.extract(r'^(?:SIRET[_:])?(\d{14})$', expand=False)
        candidat = normaliser_siret_pour_noms(extrait)
        conflit = resultat.notna() & candidat.notna() & resultat.ne(candidat)
        if conflit.any():
            raise ValueError('Les SIRET des points et de leurs identifiants se contredisent.')
        resultat = resultat.fillna(candidat)
    return resultat


def inspecter_csv_pour_noms(fichier, membre=None):
    fichier = Path(fichier)
    with ExitStack() as pile:
        if membre is None:
            flux = pile.enter_context(fichier.open('rb'))
        else:
            archive = pile.enter_context(ZipFile(fichier))
            flux = pile.enter_context(archive.open(membre))
        ligne = flux.readline()
    try:
        entete = ligne.decode('utf-8-sig')
        encodage = 'utf-8-sig'
    except UnicodeDecodeError:
        entete = ligne.decode('cp1252')
        encodage = 'cp1252'
    essais = [(sep, next(csv.reader([entete], delimiter=sep))) for sep in [';', ',', '\t']]
    separateur, colonnes = max(essais, key=lambda resultat: len(resultat[1]))
    correspondance = {c: str(c).strip().upper() for c in colonnes}
    if len(correspondance) != len(colonnes) or len(set(correspondance.values())) != len(colonnes):
        raise ValueError(f'Entête avec colonnes répétées : {fichier.name}')
    return {'fichier': fichier, 'membre': membre, 'sep': separateur,
            'encoding': encodage, 'colonnes': correspondance}


def morceaux_csv_pour_noms(source, colonnes):
    selection = [c for c, normalise in source['colonnes'].items() if normalise in colonnes]
    with ExitStack() as pile:
        if source['membre'] is None:
            flux = pile.enter_context(source['fichier'].open('rb'))
        else:
            archive = pile.enter_context(ZipFile(source['fichier']))
            flux = pile.enter_context(archive.open(source['membre']))
        lecteur = pile.enter_context(pd.read_csv(
            flux, sep=source['sep'], encoding=source['encoding'], dtype='string',
            usecols=selection, chunksize=20000, low_memory=True))
        for morceau in lecteur:
            yield morceau.rename(columns=source['colonnes'])


def trouver_bpe_pour_noms(fichier_bpe=None, dossier_bpe=None):
    # Inspecter les entêtes, sans charger les millions de lignes du fichier national.
    if fichier_bpe is not None and Path(fichier_bpe).is_file():
        fichiers = [Path(fichier_bpe)]
    elif dossier_bpe is not None and Path(dossier_bpe).is_dir():
        fichiers = sorted(p for p in Path(dossier_bpe).rglob('*')
                          if p.is_file() and p.suffix.lower() in {'.csv', '.zip'})
    else:
        return None
    valides = []
    for fichier in fichiers:
        if fichier.suffix.lower() == '.zip':
            with ZipFile(fichier) as archive:
                membres = [m for m in archive.namelist() if m.lower().endswith('.csv')]
        elif fichier.suffix.lower() == '.csv':
            membres = [None]
        else:
            raise ValueError('La récupération des noms attend une BPE au format CSV ou ZIP.')
        for membre in membres:
            source = inspecter_csv_pour_noms(fichier, membre)
            if {'SIRET', 'NOMRS', 'TYPEQU'}.issubset(source['colonnes'].values()):
                valides.append(source)
    if len(valides) > 1:
        choix = '\n'.join(str(s['fichier']) + (' : ' + s['membre'] if s['membre'] else '')
                          for s in valides)
        raise ValueError('Plusieurs sources BPE avec noms existent. Définir FICHIER_BPE_NOMS '
                         'au bloc 3 avec le fichier BPE 2025 utilisé au J6 :\n' + choix)
    return valides[0] if valides else None


def libelles_par_siret(source, cibles, colonne_libelle=None):
    disponibles = set(source['colonnes'].values())
    champs = set(COLONNES_NOMS_RESTAURANTS) | {'SIRET', 'TYPEQU', 'AN'}
    if colonne_libelle:
        champs.add(str(colonne_libelle).strip().upper())
    if 'SIRET' not in disponibles or not (champs - {'SIRET', 'TYPEQU', 'AN'}) & disponibles:
        return pd.DataFrame(columns=['LIBELLE', 'CHAMP']).rename_axis('SIRET')
    correspondances = {}
    ambigus = set()
    siret_trouves = set()
    for morceau in morceaux_csv_pour_noms(source, champs):
        if 'TYPEQU' in morceau.columns:
            morceau = morceau[morceau['TYPEQU'].str.strip().str.upper().eq('A504')].copy()
        if 'AN' in morceau.columns:
            morceau = morceau[morceau['AN'].str.strip().eq('2025')].copy()
        cles = normaliser_siret_pour_noms(morceau['SIRET'])
        morceau = morceau.loc[cles.isin(cibles)].copy()
        cles = cles.loc[morceau.index]
        siret_trouves.update(cles.dropna())
        noms, origines = choisir_noms_restaurants(morceau, colonne_libelle)
        for siret, libelle, champ in zip(cles, noms, origines):
            if pd.isna(libelle) or siret in ambigus:
                continue
            if siret in correspondances and correspondances[siret]['LIBELLE'] != libelle:
                ambigus.add(siret)
                del correspondances[siret]
            else:
                correspondances[siret] = {'LIBELLE': libelle, 'CHAMP': champ}
    print('Source de libellés :', source['fichier'], source['membre'] or '')
    print('SIRET trouvés :', len(siret_trouves), '/', len(cibles),
          '— avec libellé unique :', len(correspondances), '— ambigus :', len(ambigus))
    return pd.DataFrame.from_dict(correspondances, orient='index',
                                 columns=['LIBELLE', 'CHAMP']).rename_axis('SIRET')


def preparer_noms_restaurants(points, fichier_detail=None, fichier_bpe=None,
                             dossier_bpe=None, colonne_libelle=None):
    if 'ID_OBSERVATION' not in points.columns:
        raise ValueError('La couche des restaurants doit conserver ID_OBSERVATION.')
    resultat = points.copy()
    ids = resultat['ID_OBSERVATION'].astype('string').str.strip().replace('', pd.NA)
    if ids.isna().any() or not ids.is_unique:
        raise ValueError('Identifiants des restaurants absents ou répétés.')
    noms, colonnes = choisir_noms_restaurants(resultat, colonne_libelle)
    origines = ('POINTS:' + colonnes).astype('string')
    sirets = siret_des_restaurants(resultat)
    sources_noms = []

    if noms.isna().any() and fichier_detail is not None and Path(fichier_detail).is_file():
        source = inspecter_csv_pour_noms(fichier_detail)
        sources_noms.append(str(source['fichier']))
        disponibles = set(source['colonnes'].values())
        if 'ID_OBSERVATION' in disponibles:
            champs = set(COLONNES_NOMS_RESTAURANTS) | {'ID_OBSERVATION', 'SIRET', 'CLE_EQUIPEMENT'}
            if colonne_libelle:
                champs.add(str(colonne_libelle).strip().upper())
            morceaux = []
            for morceau in morceaux_csv_pour_noms(source, champs):
                morceau['ID_OBSERVATION'] = morceau['ID_OBSERVATION'].str.strip()
                morceau = morceau[morceau['ID_OBSERVATION'].isin(set(ids))].copy()
                if not morceau.empty:
                    morceaux.append(morceau)
            if morceaux:
                detail = pd.concat(morceaux, ignore_index=True)
                if detail['ID_OBSERVATION'].duplicated().any():
                    raise ValueError('Identifiants répétés dans le fichier détaillé des restaurants.')
                detail = detail.set_index('ID_OBSERVATION', drop=False)
                noms_csv, champs_csv = choisir_noms_restaurants(detail, colonne_libelle)
                cles_csv = ids.map(siret_des_restaurants(detail)).astype('string')
                conflit = sirets.notna() & cles_csv.notna() & sirets.ne(cles_csv)
                if conflit.any():
                    raise ValueError('Les SIRET du fichier détaillé ne correspondent pas aux points.')
                sirets = sirets.fillna(cles_csv)
                completer = noms.isna()
                noms.loc[completer] = ids.map(noms_csv).loc[completer]
                origines.loc[completer] = ('CSV:' + source['fichier'].name + ':'
                                          + ids.map(champs_csv).loc[completer])
        elif 'SIRET' in disponibles:
            correspondances = libelles_par_siret(source, set(sirets.dropna()), colonne_libelle)
            completer = noms.isna()
            noms.loc[completer] = sirets.map(correspondances['LIBELLE']).loc[completer]
            origines.loc[completer] = ('CSV:' + source['fichier'].name + ':'
                                      + sirets.map(correspondances['CHAMP']).loc[completer])

    if noms.isna().any() and (noms.isna() & sirets.notna()).any():
        source = trouver_bpe_pour_noms(fichier_bpe, dossier_bpe)
        if source is not None:
            sources_noms.append(str(source['fichier']))
            cibles = set(sirets.loc[noms.isna()].dropna())
            correspondances = libelles_par_siret(source, cibles, colonne_libelle)
            completer = noms.isna()
            noms.loc[completer] = sirets.map(correspondances['LIBELLE']).loc[completer]
            origines.loc[completer] = ('BPE:' + source['fichier'].name + ':'
                                      + sirets.map(correspondances['CHAMP']).loc[completer])
        else:
            print('Aucune BPE CSV/ZIP contenant SIRET, NOMRS et TYPEQU trouvée dans', dossier_bpe)

    print('Libellés retrouvés :', int(noms.notna().sum()), '/', len(resultat))
    if len(resultat) and not noms.notna().any():
        if not sirets.notna().any():
            raison = ('Les points et le fichier détaillé ne fournissent aucun SIRET utilisable. '
                      'Conserver SIRET et NOMRS dans les exports de restaurants J9b/J11, '
                      'ou fournir leur correspondance avec ID_OBSERVATION.')
        else:
            raison = ('Vérifier le chemin de la BPE 2025 au bloc 3 et ses valeurs NOMRS/CNOMRS. '
                      'Un SIRET peut être absent de la source, sans libellé ou associé à des noms ambigus.')
        raise ValueError('Aucun libellé de restaurant retrouvé. ' + raison
                         + '\nColonnes des points : ' + str(points.columns.tolist()))
    resultat['SIRET_JOINTURE_NOMS'] = sirets
    resultat['LIBELLE_RESTAURANT'] = noms
    resultat['NOM_RESTAURANT_RENSEIGNE'] = noms.notna()
    resultat['NOM_RESTAURANT_CARTE'] = noms.fillna('Libellé absent dans la source')
    resultat['ORIGINE_NOM_CARTE'] = origines.fillna('NON_DISPONIBLE')
    resultat.attrs['sources_libelles_restaurants'] = list(dict.fromkeys(sources_noms))
    return resultat


def ajouter_observations(fig, points, nom='Restaurants — BPE 2025',
                         couleur='#c2410c', visible=True, afficher_noms=True):
    if points.empty:
        print('Aucune observation géolocalisée à afficher :', nom)
        return 0
    exiger_colonnes(points, ['ID_OBSERVATION', 'CODGEO'], nom)
    points = points.to_crs(4326)
    noms, _ = choisir_noms_restaurants(points)
    if not noms.notna().any():
        raise ValueError('Aucun libellé retrouvé dans ce secteur. Vérifier la source des noms avant la carte.')
    libelles = []
    noms_affiches = []
    statuts = [c for c in points.columns if c.startswith('STATUT')]
    for index, ligne in points.iterrows():
        disponible = pd.notna(noms.loc[index])
        libelle = str(noms.loc[index]) if disponible else 'Libellé absent dans la source'
        noms_affiches.append(escape(libelle) if disponible else '')
        lignes = [f'<b>{escape(libelle)}</b>',
                  'Code commune déclaré : ' + escape(texte(ligne['CODGEO'])),
                  'ID observation : ' + escape(texte(ligne['ID_OBSERVATION']))]
        if statuts:
            lignes.extend(f'{escape(c)} : {escape(texte(ligne[c]))}' for c in statuts)
        else:
            lignes.append('Contrôle géographique : consulter le diagnostic J11')
        libelles.append('<br>'.join(lignes))
    fig.add_trace(go.Scattergeo(
        lon=points.geometry.x.tolist(), lat=points.geometry.y.tolist(),
        mode='markers+text' if afficher_noms else 'markers', name=nom, visible=visible,
        ids=points['ID_OBSERVATION'].astype('string').tolist(),
        marker={'size': 6, 'color': couleur, 'opacity': 0.8},
        text=noms_affiches, textposition='top center',
        textfont={'size': 10, 'color': '#7c2d12'},
        customdata=[[libelle] for libelle in libelles],
        hovertemplate='%{customdata[0]}<extra></extra>',
    ))
    return len(points)


In [ ]:
#Charger et vérifier les communes

communes = preparer_communes(lire_couche(
    FICHIER_COMMUNES, 'Communes J11', ['Polygon', 'MultiPolygon'], layer='communes'))
print('Nombre de communes :', len(communes))
print('Colonnes disponibles :')
print(communes.columns.tolist())
display(communes[['CODGEO', 'NOM_CARTE']].head())


In [ ]:
#Fonction de carte statistique

def carte_indicateur(communes, colonne, titre, unite, couleurs='YlOrRd'):
    exiger_colonnes(communes, [colonne], 'Profil communal')
    valeurs = nombres(communes[colonne])
    disponibles = valeurs.notna()
    if not disponibles.any():
        raise ValueError(f'{colonne} : aucune valeur numérique disponible.')

    diagnostic = pd.DataFrame({
        'CODGEO': communes['CODGEO'], 'NOM_COMMUNE': communes['NOM_CARTE'],
        'VALEUR_SOURCE': communes[colonne], 'VALEUR_CARTOGRAPHIEE': valeurs,
        'VALEUR_DISPONIBLE': disponibles,
    })
    fig = go.Figure()
    ajouter_contours(fig, communes, couleur='#e5e7eb')

    if (~disponibles).any():
        manquantes = communes.loc[~disponibles]
        fig.add_trace(go.Choropleth(
            geojson=contours_geojson(manquantes),
            locations=manquantes['CODGEO'].tolist(),
            featureidkey='properties.CODGEO', z=[0] * len(manquantes),
            zmin=0, zmax=1, colorscale=[[0, '#e5e7eb'], [1, '#e5e7eb']],
            showscale=False, showlegend=True, name='Valeur non disponible',
            marker_line_color='#64748b', marker_line_width=0.5,
            text=[escape(texte(n)) for n in manquantes['NOM_CARTE']],
            hovertemplate='%{text}<br>Code : %{location}'
                          '<br>Valeur non disponible<extra></extra>',
        ))

    valides = communes.loc[disponibles]
    mini, maxi = float(valeurs.min()), float(valeurs.max())
    if mini == maxi:
        maxi = mini + max(abs(mini) * 0.01, 1)
    fig.add_trace(go.Choropleth(
        geojson=contours_geojson(valides), locations=valides['CODGEO'].tolist(),
        featureidkey='properties.CODGEO', z=valeurs.loc[disponibles].tolist(),
        zmin=mini, zmax=maxi, colorscale=couleurs, showscale=True,
        showlegend=False, name=titre,
        marker_line_color='#64748b', marker_line_width=0.5,
        colorbar={'title': {'text': unite}, 'thickness': 15, 'len': 0.7},
        text=[escape(texte(n)) for n in valides['NOM_CARTE']],
        hovertemplate=('%{text}<br>Code : %{location}<br>'
                       '%{z:,.2f} ' + escape(unite) + '<extra></extra>'),
    ))
    configurer_figure(fig, titre, communes,
                     'Indicateurs : profil communal J11. Contours : référence du J11. '
                     'Gris : valeur non disponible. Géométries simplifiées pour affichage (20 m).')
    return fig, diagnostic



In [ ]:
#Choisirles indicateurs

CARTES_INDICATEURS = [
    {'colonne': 'NIVEAU_VIE_MEDIAN_2023', 'titre': 'Niveau de vie médian — 2023',
     'unite': '€/an par unité de consommation', 'couleurs': 'Blues',
     'fichier': '01_niveau_vie_median_plotly.html'},
    {'colonne': 'DENSITE_OBS_ALIMENTAIRES_B2_10000_HAB',
     'titre': 'Commerces alimentaires — densité communale',
     'unite': 'observations / 10 000 habitants', 'couleurs': 'YlOrRd',
     'fichier': '02_densite_commerces_alimentaires_plotly.html'},
    {'colonne': 'DENSITE_OBS_CULTURE_10000_HAB',
     'titre': 'Équipements culturels — densité communale',
     'unite': 'observations / 10 000 habitants', 'couleurs': 'Purples',
     'fichier': '03_densite_culture_plotly.html'},
]
display(pd.DataFrame(CARTES_INDICATEURS))


In [ ]:
#Générer les cartes d'indicateurs

bilan_cartes = []
figures_indicateurs = {}
for configuration in CARTES_INDICATEURS:
    colonne = configuration['colonne']
    if colonne not in communes.columns:
        print('Carte ignorée : colonne absente', colonne)
        bilan_cartes.append({'VARIABLE': colonne, 'ETAT': 'COLONNE_ABSENTE'})
        continue
    if not nombres(communes[colonne]).notna().any():
        print('Carte ignorée : aucune valeur numérique', colonne)
        bilan_cartes.append({'VARIABLE': colonne, 'ETAT': 'AUCUNE_VALEUR_NUMERIQUE'})
        continue
    fig, diagnostic = carte_indicateur(communes, colonne, configuration['titre'],
                                      configuration['unite'], configuration['couleurs'])
    sortie = DOSSIER_CARTES / configuration['fichier']
    afficher_et_exporter(fig, sortie)
    enregistrer_csv(diagnostic, DOSSIER_INTERIM / f'{colonne.lower()}_controle_carte.csv')
    figures_indicateurs[colonne] = fig
    bilan_cartes.append({'VARIABLE': colonne, 'ETAT': 'CREEE', 'FICHIER': str(sortie),
                        'NB_COMMUNES': len(diagnostic),
                        'NB_VALEURS_DISPONIBLES': int(diagnostic['VALEUR_DISPONIBLE'].sum()),
                        'NB_VALEURS_MANQUANTES': int((~diagnostic['VALEUR_DISPONIBLE']).sum())})
if not figures_indicateurs:
    raise ValueError('Aucune carte créée : adapter CARTES_INDICATEURS aux colonnes du profil.')
display(pd.DataFrame(bilan_cartes))


In [ ]:
#Choisir le secteur local

CODES_COMMUNES = ['95127', '95500']
MARGE_AFFICHAGE_METRES = 3000
communes_selection, zone_affichage, communes_contexte = selectionner_secteur(
    communes, CODES_COMMUNES, MARGE_AFFICHAGE_METRES)
display(communes_selection[['CODGEO', 'NOM_CARTE']])


In [ ]:
#Charger les points controlés des resturants

restaurants = lire_couche(FICHIER_RESTAURANTS, 'Restaurants J11', ['Point'])
exiger_colonnes(restaurants, ['ID_OBSERVATION', 'CODGEO'], 'Restaurants')
if restaurants['ID_OBSERVATION'].isna().any() or not restaurants['ID_OBSERVATION'].is_unique:
    raise ValueError('Identifiants des observations absents ou répétés.')
restaurants = preparer_noms_restaurants(
    restaurants, fichier_detail=FICHIER_DETAIL_RESTAURANTS,
    fichier_bpe=FICHIER_BPE_NOMS, dossier_bpe=DOSSIER_BPE_NOMS,
    colonne_libelle=COLONNE_LIBELLE_RESTAURANT)
display(restaurants[['ID_OBSERVATION', 'SIRET_JOINTURE_NOMS', 'LIBELLE_RESTAURANT',
                     'ORIGINE_NOM_CARTE']].head(10))
restaurants_locaux = restaurants[
    restaurants.geometry.intersects(zone_affichage.geometry.iloc[0])].copy()
print('Observations géolocalisées dans l’emprise :', len(restaurants_locaux))
print('Statuts disponibles :', [c for c in restaurants.columns if c.startswith('STATUT')])
enregistrer_csv(pd.DataFrame(restaurants_locaux[['ID_OBSERVATION', 'CODGEO',
    'SIRET_JOINTURE_NOMS', 'LIBELLE_RESTAURANT', 'NOM_RESTAURANT_CARTE',
    'NOM_RESTAURANT_RENSEIGNE', 'ORIGINE_NOM_CARTE']]),
    DOSSIER_INTERIM / 'noms_restaurants_carte.csv')



In [ ]:
#Carte locale des restaurants

carte_restaurants = go.Figure()
ajouter_contours(carte_restaurants, communes_contexte)
ajouter_contours(carte_restaurants, communes_selection, couleur='rgba(0,0,0,0)', largeur=1.5)
nb_points_affiches = ajouter_observations(carte_restaurants, restaurants_locaux,
                                         afficher_noms=AFFICHER_NOMS_RESTAURANTS)
configurer_figure(carte_restaurants, 'Cergy et Pontoise — restaurants et contexte communal',
                 zone_affichage, 'Source : BPE 2025, contrôle géographique J11. '
                 'Marge de 3 km pour affichage. Points sans coordonnées absents de cette carte.')
FICHIER_CARTE_RESTAURANTS = DOSSIER_CARTES / '04_restaurants_cergy_pontoise_plotly.html'
afficher_et_exporter(carte_restaurants, FICHIER_CARTE_RESTAURANTS)




In [ ]:
for colonne, fig in figures_indicateurs.items():
    traces_valeurs = [t for t in fig.data if t.type == 'choropleth' and t.showscale]
    assert len(traces_valeurs) == 1
    assert len(traces_valeurs[0].locations) == int(nombres(communes[colonne]).notna().sum())
    assert fig.layout.geo.visible is False
    assert all(t.type in ['choropleth', 'scattergeo'] for t in fig.data)
assert nb_points_affiches == len(restaurants_locaux)
assert carte_restaurants.layout.geo.visible is False
fichiers_cartes = [Path(ligne['FICHIER']) for ligne in bilan_cartes if ligne['ETAT'] == 'CREEE']
fichiers_cartes.append(FICHIER_CARTE_RESTAURANTS)
for fichier in fichiers_cartes:
    assert fichier.is_file() and fichier.stat().st_size > 0
enregistrer_csv(pd.DataFrame(bilan_cartes), DOSSIER_INTERIM / 'bilan_cartes_plotly.csv')
metadata = {
    'etape': 'J11_CARTOGRAPHIE_PLOTLY',
    'date_execution_utc': datetime.now(timezone.utc).isoformat(),
    'plotly_version': plotly.__version__, 'fond_de_tuiles': False, 'cle_api_requise': False,
    'crs_calculs': 'EPSG:2154', 'crs_affichage': 'EPSG:4326',
    'simplification_contours_affichage_metres': 20,
    'codes_communes_etudiees': CODES_COMMUNES,
    'marge_affichage_metres': MARGE_AFFICHAGE_METRES,
    'afficher_noms_restaurants': AFFICHER_NOMS_RESTAURANTS,
    'nb_restaurants_locaux_avec_nom': int(restaurants_locaux['NOM_RESTAURANT_RENSEIGNE'].sum()),
    'fichier_detail_noms': str(FICHIER_DETAIL_RESTAURANTS),
    'fichier_bpe_noms_configure': str(FICHIER_BPE_NOMS),
    'sources_libelles_restaurants': restaurants.attrs.get('sources_libelles_restaurants', []),
    'sources': [{'fichier': str(p), 'sha256': empreinte(p)}
                for p in [FICHIER_COMMUNES, FICHIER_RESTAURANTS]
                + [Path(p) for p in restaurants.attrs.get('sources_libelles_restaurants', [])]],
    'cartes': [str(p) for p in fichiers_cartes],
    'limites': ['Les observations sans coordonnées ne figurent pas sur la carte de points.',
                'La marge ne représente pas une accessibilité piétonne.',
                'La localisation et l’ouverture actuelle nécessitent une vérification terrain.'],
}
(DOSSIER_INTERIM / 'metadata_cartes_plotly.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Cartes créées. Vérifier visuellement les couleurs, légendes et localisations.')
